In [ ]:
import pandas as pd

df = pd.read_csv("email_spam.csv", encoding="latin-1")

# Keep only required columns
df = df[['v1','v2']]

# Rename columns for clarity
df.columns = ['label','message']

df.head()

In [ ]:
df.shape


In [ ]:
df['label'].value_counts()
import matplotlib.pyplot as plt

df['label'].value_counts().plot(kind='bar')

plt.title("Spam vs Ham Distribution")
plt.xlabel("Message Type")
plt.ylabel("Number of Messages")

plt.show()


In [ ]:
df.isnull().sum()

In [ ]:
df['length'] = df['message'].apply(len)

df.groupby('label')['length'].mean()

df.hist(column='length', by='label', bins=50)
plt.show()

In [ ]:
import numpy as np
import re
import nltk
import string

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

In [ ]:
df.head()

In [ ]:
df['message'] = df['message'].str.lower()

In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['message'] = df['message'].apply(remove_punctuation)

In [ ]:
df['message'] = df['message'].apply(lambda x: re.sub(r'\d+', '', x))

In [ ]:
df['tokens'] = df['message'].apply(lambda x: x.split())

In [ ]:
stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(
    lambda words: [word for word in words if word not in stop_words]
)

In [ ]:
stemmer = PorterStemmer()

df['tokens'] = df['tokens'].apply(
    lambda words: [stemmer.stem(word) for word in words]
)

In [ ]:
df['clean_message'] = df['tokens'].apply(lambda words: " ".join(words))

In [ ]:
df[['message','clean_message']].head()

In [ ]:
X_clean = df['clean_message']
y = df['label']

In [ ]:
!pip install transformers

In [ ]:
import torch
from transformers import DistilBertModel, DistilBertTokenizer
import numpy as np

# Load pre-trained model and tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

def extract_distilbert_features(text_list):
    model.eval()
    features = []

    for text in text_list:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)

        # Get the [CLS] token embedding (index 0)
        cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
        features.append(cls_embedding.flatten())

    return np.array(features)

# Applying extraction to the cleaned dataset
X_features = extract_distilbert_features(df['clean_message'].tolist())

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

models = {

    "Hybrid_SVM": SVC(kernel='linear', probability=True),

    "Hybrid_RandomForest":
    RandomForestClassifier(n_estimators=100),

    "Hybrid_GradientBoost":
    GradientBoostingClassifier(),

    "Hybrid_LogisticRegression":
    LogisticRegression(max_iter=1000),

    "Hybrid_KNN":
    KNeighborsClassifier(n_neighbors=5),

    "Hybrid_DecisionTree":
    DecisionTreeClassifier(),

    "Hybrid_ExtraTrees":
    ExtraTreesClassifier(n_estimators=100),

    "Hybrid_AdaBoost":
    AdaBoostClassifier(),

    "Hybrid_GaussianNB":
    GaussianNB(),

    "Hybrid_MLP":
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
}

In [ ]:
from sklearn.model_selection import train_test_split

# Split data (80% Training, 20% Testing)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# Training Loop
for name, model_obj in models.items():
    print(f"Training {name}...")
    model_obj.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

evaluation_results = {}

for name, model_obj in models.items():

    predictions = model_obj.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    report = classification_report(
        y_test,
        predictions,
        output_dict=True
    )

    evaluation_results[name] = {
        "Accuracy": accuracy,
        "Precision": report['1']['precision'],   # spam class
        "Recall": report['1']['recall'],
        "F1-score": report['1']['f1-score']
    }

    print(f"\n--- {name} Results ---")

    print("Confusion Matrix:\n",
          confusion_matrix(y_test, predictions))

    print("Accuracy:", accuracy)
    print("Precision:", report['1']['precision'])
    print("Recall:", report['1']['recall'])
    print("F1-score:", report['1']['f1-score'])

# Convert results to table
results_df = pd.DataFrame(evaluation_results).T

results_df

In [ ]:
import joblib
from transformers import DistilBertTokenizer, DistilBertModel

# Select best model using F1-score
best_model_name = max(
    evaluation_results,
    key=lambda x: evaluation_results[x]['F1-score']
)

best_model = models[best_model_name]

# Save trained classifier
joblib.dump(best_model, "best_spam_model.pkl")

# Save tokenizer (needed during prediction)
tokenizer.save_pretrained("distilbert_tokenizer")

print(f"Selected Model: {best_model_name}")
print("Best model and tokenizer saved successfully.")

In [ ]:
!pip install flask transformers torch pyngrok

In [ ]:
# ==========================================================
# 1. INSTALL REQUIRED LIBRARIES
# ==========================================================
!pip install flask transformers torch joblib

# ==========================================================
# 2. IMPORT LIBRARIES
# ==========================================================
import os
import joblib
import torch
from flask import Flask, request, render_template
from transformers import DistilBertTokenizer, DistilBertModel
from google.colab.output import eval_js

# ==========================================================
# 3. CREATE FRONTEND TEMPLATE
# ==========================================================
if not os.path.exists("templates"):
    os.makedirs("templates")

html_content = """
<!DOCTYPE html>
<html>
<head>
<title>Spam Detection System</title>

<style>

body{
font-family:Arial;
background:#eef2f7;
display:flex;
justify-content:center;
align-items:center;
height:100vh;
}

.card{
background:white;
padding:40px;
border-radius:15px;
box-shadow:0 10px 25px rgba(0,0,0,0.1);
width:500px;
text-align:center;
}

textarea{
width:100%;
height:120px;
padding:15px;
border-radius:8px;
border:1px solid #ddd;
}

button{
margin-top:20px;
padding:12px 30px;
border:none;
background:#4A90E2;
color:white;
border-radius:25px;
cursor:pointer;
}

.result{
margin-top:20px;
font-size:18px;
font-weight:bold;
}

</style>

</head>

<body>

<div class="card">

<h2>Spam Detection System</h2>

<p>Hybrid DistilBERT + Machine Learning Classifier</p>

<form action="/predict" method="post">

<textarea name="message"
placeholder="Enter SMS or Email text here..."
required></textarea>

<br>

<button type="submit">Check Message</button>

</form>

<div class="result">

{{ prediction_text }}

</div>

</div>

</body>
</html>
"""

with open("templates/index.html", "w") as f:
    f.write(html_content)

# ==========================================================
# 4. LOAD TRAINED MODELS
# ==========================================================
print("Loading classifier and DistilBERT...")

model_ml = joblib.load("best_spam_model.pkl")

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

bert_model = DistilBertModel.from_pretrained(
    "distilbert-base-uncased"
)

bert_model.eval()

print("Models loaded successfully!")

# ==========================================================
# 5. CREATE FLASK APP
# ==========================================================
app = Flask(__name__)

@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():

    message = request.form["message"]

    # DistilBERT embedding extraction
    inputs = tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state[:,0,:].cpu().numpy()

    # ML classifier prediction
    prediction = model_ml.predict(embedding)

    result = "SPAM" if prediction[0]==1 else "LEGITIMATE (HAM)"

    return render_template(
        "index.html",
        prediction_text=f"This message is: {result}"
    )

# ==========================================================
# 6. RUN WEB APPLICATION INSIDE COLAB
# ==========================================================
print("Click the link below to open the web app:")

print(eval_js("google.colab.kernel.proxyPort(5000)"))

app.run(port=5000)